# Deliverable 3 — Public API Data Extraction

**Purpose:** retrieve cryptocurrency market prices from CoinGecko's public API, inspect the JSON response, flatten it into a table, and save it as CSV.

**Endpoint:** `https://api.coingecko.com/api/v3/simple/price?ids=bitcoin,ethereum&vs_currencies=usd,inr`

**Note:** Public API access/rate limits can change. Run the notebook with internet access. If the API rejects a request, wait and retry later. The included CSV is a clearly labelled example snapshot to show the expected structure; the notebook generates a fresh CSV when executed.

In [1]:
import requests
import pandas as pd
from datetime import datetime, timezone
from pathlib import Path

url = 'https://api.coingecko.com/api/v3/simple/price'
params = {'ids': 'bitcoin,ethereum', 'vs_currencies': 'usd,inr'}
response = requests.get(url, params=params, timeout=30)
response.raise_for_status()
data = response.json()
data

{'bitcoin': {'usd': 75973, 'inr': 7179019}, 'ethereum': {'usd': 2248.79, 'inr': 212499}}

## Inspect the JSON
The response is a nested dictionary: each coin is a key, and its USD/INR prices are fields inside that key.

In [2]:
print('Top-level type:', type(data).__name__)
for coin, prices in data.items():
    print(coin, '->', prices)

Top-level type: dict
bitcoin -> {'usd': 75973, 'inr': 7179019}
ethereum -> {'usd': 2248.79, 'inr': 212499}


## Convert JSON to a flat table and export CSV

In [3]:
rows = []
extracted_at = datetime.now(timezone.utc).isoformat(timespec='seconds')
for coin, prices in data.items():
    rows.append({
        'asset_id': coin,
        'price_usd': prices.get('usd'),
        'price_inr': prices.get('inr'),
        'extracted_at_utc': extracted_at
    })

df = pd.DataFrame(rows)
display(df)
df.to_csv('coingecko_prices.csv', index=False)
print('Saved coingecko_prices.csv')

,asset_id,price_usd,price_inr,extracted_at_utc
0,bitcoin,75973.00,7179019,2026-11-05T19:27:06+00:00
1,ethereum,2248.79,212499,2026-11-05T19:27:06+00:00


Saved coingecko_prices.csv


## Findings and limitations
- The API returned one record per requested asset.
- The nested JSON was flattened into columns for asset ID, USD price, INR price, and extraction timestamp.
- This endpoint provides a point-in-time quote, not a complete historical price series.
- Cryptocurrency prices are volatile. This exercise is for API and data-handling practice, not investment advice.
- CoinGecko may enforce rate limits; do not repeatedly call the endpoint in a tight loop.

In [4]:
# Basic validation
assert not df.empty, 'No records were returned.'
assert {'asset_id', 'price_usd', 'price_inr', 'extracted_at_utc'}.issubset(df.columns)
assert df[['price_usd', 'price_inr']].notna().all().all()
print('Validation passed:', len(df), 'assets extracted.')

Validation passed: 2 assets extracted.
